In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

In [8]:
tokenizer = AutoTokenizer.from_pretrained("models/")
model = AutoModelForTokenClassification.from_pretrained("models/")

label_list = ["O", "B-SYMPTOM", "I-SYMPTOM", "B-DURATION", "I-DURATION"]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7857.53it/s]


In [9]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)[0]

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    results = []
    for token, pred in zip(tokens, preds):
        if token not in ["[CLS]", "[SEP]", "[PAD]"]:
            results.append((token, label_list[pred]))

    return results

In [10]:
def clean_predictions(predictions):
    stopwords = {
        "i", "me", "my", "and", "or", "but",
        "the", "a", "is", "was", "are",
        "day", "after", "before"
    }

    cleaned = []

    for word, label in predictions:
        if word in stopwords:
            continue

        if len(word) <= 2:
            continue

        cleaned.append((word, label))

    return cleaned

In [11]:
def enhance_predictions(predictions):
    enhanced = []

    for i, (word, label) in enumerate(predictions):

        # detect pain phrases
        if word in ["pain", "hurt", "hurts"]:
            if i > 0:
                enhanced.append((predictions[i-1][0], "B-SYMPTOM"))
                enhanced.append((word, "I-SYMPTOM"))
                continue

        # breathing issues
        if word in ["breath", "breathing"]:
            enhanced.append((word, "B-SYMPTOM"))
            continue

        # movement issues
        if word in ["move", "moving"]:
            enhanced.append((word, "B-SYMPTOM"))
            continue

        enhanced.append((word, label))

    return enhanced

In [12]:
text = "I feel tensed and have stomach pain since yesterday"
output = predict(text)

for word, label in output:
    print(word, "→", label)

i → O
feel → O
tensed → B-SYMPTOM
and → O
have → O
stomach → B-SYMPTOM
pain → I-SYMPTOM
since → O
yesterday → O


In [13]:
def to_structured_output(predictions):
    symptoms = []
    current = []

    for word, label in predictions:

        if "SYMPTOM" in label:
            current.append(word)
        else:
            if current:
                symptoms.append(" ".join(current))
                current = []

    if current:
        symptoms.append(" ".join(current))

    return {
        "symptoms": symptoms,
        "duration": ""
    }

In [14]:
structured = to_structured_output(output)

print("\nFinal Structured Output:")
print(structured)


Final Structured Output:
{'symptoms': ['tensed', 'stomach pain'], 'duration': ''}
